In [2]:
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

In [7]:
files = fs.find("s3://mateomorin/legifrance/documents/2017", detail=True)

In [12]:
import pandas as pd

file_sizes = []

for path, info in files.items():
    if info["type"] == "file" and info["size"] >= 1e6:
        file_sizes.append({"file": path, "size_mo": round(info["size"]/(1e6), 2)})

df_sizes = pd.DataFrame(file_sizes)

In [17]:
df_sizes.sort_values("size_mo", ascending=False).head(10)["file"].to_list()

['mateomorin/legifrance/documents/2017/11/ACCOTEXT000036508884.docx',
 'mateomorin/legifrance/documents/2017/10/ACCOTEXT000036676416.docx',
 'mateomorin/legifrance/documents/2017/11/ACCOTEXT000036595833.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000036595804.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037151099.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037170501.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000036745268.docx',
 'mateomorin/legifrance/documents/2017/11/ACCOTEXT000036508797.docx',
 'mateomorin/legifrance/documents/2017/09/ACCOTEXT000036723314.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037908855.docx']

In [4]:
import os

from dotenv import load_dotenv
from markitdown import MarkItDown
from openai import OpenAI

load_dotenv(override=True)

llm_client = OpenAI(
    base_url=os.environ["LLM_API_URL"],
    api_key=os.environ["LLM_API_KEY"],
)

md = MarkItDown(
    enable_plugins=True,
    llm_client=llm_client,
    llm_model="qwen3-8-27b",
    llm_prompt="Ecris tout le texte que tu vois sur l'image, en préservant la structure des paragraphes et des titres. Veille à respecter l'ortographe, la syntaxe et la police de caractère (gras, itallique, ...). TOUT DOIT ÊTRE ECRIT EN FORMAT MARKDOWN",
)

In [5]:
result = md.convert("ACCOTEXT000036508884.pdf")

In [7]:
basic_md = MarkItDown(
    enable_plugins=False,
)

In [10]:
import docx

In [16]:
doc = docx.Document("ACCOTEXT000036508884.docx")

In [18]:
count = 0

for par in doc.paragraphs:
    if 'graphicData' in par._p.xml or 'imagedata' in par._p.xml:
        count += 1

print(count)

42


In [9]:
basic_md.convert("ACCOTEXT000036508884.docx").markdown

'![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-em

In [6]:
with open("ACCOTEXT000036508884_qwen3-8-27b.md", "w") as f:
    f.write(result.markdown)

In [ ]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)
metadata = pd.read_parquet("s3://mateomorin/legifrance/metadata/acco_metadata_2018.parquet", filesystem=fs)